In [1]:
import numpy as np
import dimod
import matplotlib.pyplot as plt
from itertools import product
import yfinance as yf

In [2]:
n_assets=5
n_bits=3       #bits per asset weight (gives us 2^3=8)
N=n_assets*n_bits #total binary variables

"""
Class below builds the QUBO matrix Q which has all the portfolio optimization problems.
We want to minimize x^TQ^x where x is the binary vector

What we encode: return maximization, risk minimization, budget constraints, sector cap constraint.

Parameters:
mean_returns : array of expected returns
cov_matrix: covariance matrix
lambda_budget: penalty coefficient for budget constraint
lambda_sector: penalty coefficient for sector cap constraint
sector_map : dictionary mapping asset index to its sector
sector_cap: max budget allocation per sector
"""
def build_qubo_matrix(mean_returns,cov_matrix,lambda_risk,lambda_budget,lambda_sector,sector_map,sector_cap):
    Q = np.zeros((N,N))

    #Returns term: maximize return= minimize negative return
    for i in range(n_assets):
        for k in range(n_bits):
            idx = i * n_bits + k
            weight_contribution = (2**k) /(2**n_bits -1)
            Q[idx,idx] -= mean_returns[i] * weight_contribution

    #Risk term: lambda_risk * w^T * sigma * w converted to binary
    for i in range(n_assets):
        for j in range(n_assets):
            for k in range(n_bits):
                for m in range(n_bits):
                    row = i * n_bits + k
                    col = j * n_bits + m
                    wi = (2**k) /(2**n_bits -1)
                    wj = (2**m)/(2**n_bits -1)
                    Q[row,col] += lambda_risk * cov_matrix[i,j] * wi * wj

    #Budget constraints: (sum_i w_i -1)^2 expanded into binary using binary idempotency so x_i^2=x_i
    for i in range(n_assets):
        for k in range(n_bits):
            idx= i *n_bits + k
            wi = (2**k)/ (2**n_bits-1)
            Q[idx,idx] += lambda_budget * wi * (wi-2) # diagonal terms

            for j in range(n_assets):
                for m in range(n_bits):
                    col = j*n_bits + m
                    if col> idx:
                        wj = (2**m)/(2**n_bits -1)
                        Q[idx,col] += 2 * lambda_budget * wi * wj

    # Sector cap constraint: upstream penalty
    # implemented as a quadratic penalty added to Q
    # for each sector s: (sum_{i in s} w_i - cap_s)^2 if sum > cap_s, else 0
    sectors = set(sector_map.values())
    for sector in sectors:
        sector_assets = [i for i, s in sector_map.items() if s == sector]
        for i in sector_assets:
            for k in range(n_bits):
                idx = i*n_bits +k
                wi = (2**k)/(2**n_bits -1)
                Q[idx,idx] += lambda_sector * wi * (wi -2 * sector_cap)
                for j in sector_assets:
                    for m in range(n_bits):
                        col= j*n_bits + m
                        if col>idx:
                            wj = (2**m)/(2**n_bits-1)
                            Q[idx,col]+= 2 * lambda_sector * wi * wj
    return Q
            

In [3]:
tickers = ["AAPL", "MSFT","GOOGL","JPM","GS"]
data= yf.download(tickers, start = "2020-01-01", end = "2026-01-01")
prices=data["Close"]
log_returns=np.log(prices/prices.shift(1)).dropna()

mean_returns = log_returns.mean().values * 252
cov_matrix = log_returns.cov().values * 252

# scetor map: 0=tech, 1=finance. ticker:sector
sector_map = {0:0,1:0,2:0,3:1,4:1}

Q= build_qubo_matrix(mean_returns=mean_returns,cov_matrix=cov_matrix,lambda_risk=1.0,lambda_budget=5.0
                    ,lambda_sector=2.0,sector_map=sector_map,sector_cap=0.6)
print(f"Q matrix shape: {Q.shape}")


[*********************100%***********************]  5 of 5 completed

Q matrix shape: (15, 15)


In [4]:

# Brute force (using small N s.t n_assets=3,n_bits=2 for tractability)
# N = assets * bits ∴ 2^15 = 32768

def brute_force_qubo(Q): #solves by searching over all 2^N binary strings
    N = Q.shape[0]
    best_energy = np.inf
    best_x = None

    for bits in product([0,1], repeat=N):
        x=np.array(bits)
        energy = x@Q@x
        if energy < best_energy:
            best_energy = energy
            best_x = x.copy
    return best_x, best_energy

x_brute,energy_brute = brute_force_qubo(Q)
print(f"Brute force optimal energy: {energy_brute: .5f}")


Brute force optimal energy: -6.53607


In [8]:
# Simulated annealing

#convert Q matrix into dimod formal
Q_dict={}
for i in range(N):
    for j in range(i,N):
        if i == j:
            val = Q[i,j]
        else:
            val = Q[i,j] + Q[j,i] # allows to fold the lower triange in and don't discard it
        if val != 0:
            Q_dict[(i,j)] = val
            
        # if Q[i,j]!=0:
        #     Q_dict[(i,j)] = Q[i,j]

bqm = dimod.BinaryQuadraticModel.from_qubo(Q_dict)
sampler = dimod.SimulatedAnnealingSampler()
response = sampler.sample(bqm,num_reads=1000,num_sweeps=1000)

best_sample = response.first.sample
best_energy= response.first.energy

print(f"Simulated annealing optimal energy: {best_energy:.5f}")

#Return binary solution back to portfolio weights
x_sa = np.array([best_sample[i] for i in range(N)])
weights_qubo = np.array([sum((2**k)* x_sa[i*n_bits +k] for k in range(n_bits)) / (2**n_bits -1) for i in range(n_assets)])
print("\nQUBO Portfolio Weights:")
for ticker, w in zip(tickers,weights_qubo):
    print(f"{ticker}: {w:.3f}")

Simulated annealing optimal energy: -6.53607

QUBO Portfolio Weights:
AAPL: 0.000
MSFT: 0.286
GOOGL: 0.286
JPM: 0.000
GS: 0.429


SA optimal energy = Brute force optimal energy ==> SA found the true optimum

In [9]:
# Compare to the Markowitz results from the classical portfolio engine

import sys
sys.path.append('/Users/linanachdi/Documents/GitHub/portfolio-optimization-engine')
import portfolio_engine as pe

ModuleNotFoundError: No module named 'seaborn'